# InternCircle Task 5 – Streamlit Interactive ML Web App

## Objective

To develop an interactive machine learning web application using Streamlit.

The application uses a trained Decision Tree classification model to predict Titanic passenger survival based on user-provided passenger details.

The project includes model training, model saving using Joblib, Streamlit interface creation, interactive prediction, and deployment preparation.

In [4]:
!pip -q install streamlit joblib

import pandas as pd
import numpy as np
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Libraries installed and imported successfully!")

Libraries installed and imported successfully!


## 1. Load the Titanic Dataset

The Titanic dataset is loaded using Seaborn.

In [5]:
# Load Titanic dataset
df = sns.load_dataset("titanic")

print("Titanic dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Titanic dataset loaded successfully!
Dataset shape: (891, 15)

Column names:
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Select Features and Target

The selected passenger attributes are used to predict whether a passenger survived.

Target:
survived

Features:
pclass, sex, age, sibsp, parch, fare, embarked

In [6]:
features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]

target = "survived"

X = df[features].copy()
y = df[target].copy()

print("Features selected:")
print(features)

print("\nTarget selected:")
print(target)

Features selected:
['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']

Target selected:
survived


## 3. Split the Dataset

The dataset is divided into training and testing data.

80% of the data is used for training and 20% is used for testing.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (712, 7)
Testing data shape: (179, 7)


## 4. Data Preprocessing

Missing numerical values are filled using the median.

Missing categorical values are filled using the most frequent value.

Categorical features are converted into numerical form using One-Hot Encoding.

In [8]:
numeric_features = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "fare"
]

categorical_features = [
    "sex",
    "embarked"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Data preprocessing pipeline created successfully!")

Data preprocessing pipeline created successfully!


## 5. Train the Decision Tree Classifier

A Decision Tree Classifier is trained to predict passenger survival.

In [9]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Decision Tree model trained successfully!")

Decision Tree model trained successfully!


## 6. Evaluate the Machine Learning Model

The model is evaluated using accuracy, confusion matrix, and classification report.

In [10]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Model Accuracy:", round(accuracy, 4))

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.7654

Confusion Matrix:
[[97 13]
 [29 40]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.88      0.82       110
           1       0.75      0.58      0.66        69

    accuracy                           0.77       179
   macro avg       0.76      0.73      0.74       179
weighted avg       0.76      0.77      0.76       179



## 7. Save the Trained Model

The complete machine learning pipeline is saved as a Joblib file.

This file will be loaded by the Streamlit application.

In [11]:
model_file = "titanic_survival_model.pkl"

joblib.dump(model, model_file)

print("Model saved successfully!")
print("Saved file:", model_file)

Model saved successfully!
Saved file: titanic_survival_model.pkl


## 8. Test the Saved Model

The saved model is loaded again and tested with a sample passenger.

In [12]:
loaded_model = joblib.load("titanic_survival_model.pkl")

sample_passenger = pd.DataFrame({
    "pclass": [1],
    "sex": ["female"],
    "age": [30],
    "sibsp": [0],
    "parch": [0],
    "fare": [100.0],
    "embarked": ["S"]
})

prediction = loaded_model.predict(sample_passenger)[0]
probability = loaded_model.predict_proba(sample_passenger)[0][1]

print("Sample prediction:", prediction)
print("Survival probability:", round(probability * 100, 2), "%")

Sample prediction: 1
Survival probability: 100.0 %


## 9. Create the Streamlit Application

The Streamlit application provides an interactive interface for entering passenger information and receiving a real-time survival prediction.

In [13]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

# Load model
model = joblib.load("titanic_survival_model.pkl")

# Page settings
st.set_page_config(
    page_title="Titanic Survival Predictor",
    page_icon="🚢",
    layout="centered"
)

# Title
st.title("🚢 Titanic Survival Predictor")

st.write(
    "Enter passenger details to predict Titanic survival."
)

# Inputs
pclass = st.selectbox(
    "Passenger Class",
    [1, 2, 3]
)

sex = st.selectbox(
    "Gender",
    ["female", "male"]
)

age = st.number_input(
    "Age",
    min_value=1.0,
    max_value=100.0,
    value=30.0
)

sibsp = st.number_input(
    "Siblings / Spouses",
    min_value=0,
    max_value=8,
    value=0,
    step=1
)

parch = st.number_input(
    "Parents / Children",
    min_value=0,
    max_value=6,
    value=0,
    step=1
)

fare = st.number_input(
    "Fare",
    min_value=0.0,
    max_value=600.0,
    value=32.0
)

embarked = st.selectbox(
    "Port of Embarkation",
    ["S", "C", "Q"]
)

# Prediction
if st.button("Predict Survival"):

    input_data = pd.DataFrame({
        "pclass": [pclass],
        "sex": [sex],
        "age": [age],
        "sibsp": [sibsp],
        "parch": [parch],
        "fare": [fare],
        "embarked": [embarked]
    })

    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    st.subheader("Prediction Result")

    if prediction == 1:
        st.success("✅ Passenger is predicted to survive.")
    else:
        st.error("❌ Passenger is predicted not to survive.")

    st.metric(
        "Survival Probability",
        f"{probability * 100:.2f}%"
    )

    st.subheader("Passenger Details")
    st.dataframe(input_data)

Writing app.py


## 10. Create Requirements File

The requirements file contains the libraries needed to run the Streamlit application.

In [14]:
%%writefile requirements.txt

streamlit
pandas
numpy
scikit-learn
joblib

Writing requirements.txt


## 11. Check Final Project Files

The required files are checked before uploading the project to GitHub.

In [15]:
import os

required_files = [
    "app.py",
    "titanic_survival_model.pkl",
    "requirements.txt"
]

print("Project files:")

for file in required_files:
    if os.path.exists(file):
        print("✅", file)
    else:
        print("❌", file)

Project files:
✅ app.py
✅ titanic_survival_model.pkl
✅ requirements.txt


## 12. Verify the Streamlit Application

The application files are ready for deployment.

In [16]:
print("Streamlit application prepared successfully!")
print("Model file prepared successfully!")
print("Requirements file prepared successfully!")
print("Project is ready for GitHub upload and deployment.")

Streamlit application prepared successfully!
Model file prepared successfully!
Requirements file prepared successfully!
Project is ready for GitHub upload and deployment.


## 13. Conclusion

An interactive machine learning web application was developed using Streamlit.

The application uses a Decision Tree Classifier trained on the Titanic dataset. The trained model is saved using Joblib and loaded into the Streamlit application to generate real-time survival predictions from user inputs.

This project demonstrates an end-to-end machine learning workflow from data preprocessing and model training to interactive web application development.

In [17]:
print("========== CONCLUSION ==========")
print("Interactive ML Web App created successfully!")
print("Dataset: Titanic")
print("Model: Decision Tree Classifier")
print("Interface: Streamlit")
print("Model Loading: Joblib")
print("Project completed successfully!")

========== CONCLUSION ==========
Interactive ML Web App created successfully!
Dataset: Titanic
Model: Decision Tree Classifier
Interface: Streamlit
Model Loading: Joblib
Project completed successfully!
